In [1]:
import numpy as np
import csv

In [2]:
counties = ["Nairobi", "Nakuru", "Kisumu", "Mombasa", "Machakos"]

months = [
    "January", "February", "March", "April",
    "May", "June", "July", "August",
    "September", "October", "November", "December"
]

clean_rows = [
    [64, 56, 92, 219, 176, 35, 18, 24, 31, 61, 165, 104],   # Nairobi
    [35, 42, 70, 145, 120, 78, 96, 105, 72, 55, 60, 38],    # Nakuru
    [58, 90, 150, 240, 180, 95, 70, 85, 90, 88, 130, 95],   # Kisumu
    [30, 15, 55, 150, 240, 110, 90, 65, 60, 90, 100, 70],   # Mombasa
]

machakos_raw = [
    "48", "35", "88", "N/A", "140", " 20 ",
    "9", "12", "", "45", "182", "96"
]

In [3]:
def clean_machakos_data(raw_values, month_names):
    valid_values = []
    invalid_indexes = []

    for index, value in enumerate(raw_values):
        try:
            valid_values.append(int(value.strip()))
        except ValueError:
            invalid_indexes.append(index)

    imputed_value = round(sum(valid_values) / len(valid_values))

    cleaned_values = []
    for index, value in enumerate(raw_values):
        try:
            cleaned_values.append(int(value.strip()))
        except ValueError:
            cleaned_values.append(imputed_value)

    imputed_months = [month_names[index] for index in invalid_indexes]

    return cleaned_values, imputed_months, imputed_value


machakos_clean, imputed_months, imputed_value = clean_machakos_data(
    machakos_raw, months
)

print("Cleaned Machakos rainfall:", machakos_clean)
print("Imputed value:", imputed_value, "mm")
print("Imputed months:", ", ".join(imputed_months))

Cleaned Machakos rainfall: [48, 35, 88, 68, 140, 20, 9, 12, 68, 45, 182, 96]
Imputed value: 68 mm
Imputed months: April, September


In [4]:
R = np.array(clean_rows + [machakos_clean])

print("Combined rainfall array:")
print(R)

print("\nShape:", R.shape)
print("Data type:", R.dtype)

Combined rainfall array:
[[ 64  56  92 219 176  35  18  24  31  61 165 104]
 [ 35  42  70 145 120  78  96 105  72  55  60  38]
 [ 58  90 150 240 180  95  70  85  90  88 130  95]
 [ 30  15  55 150 240 110  90  65  60  90 100  70]
 [ 48  35  88  68 140  20   9  12  68  45 182  96]]

Shape: (5, 12)
Data type: int64


In [5]:
annual_totals = R.sum(axis=1)
monthly_averages = R.mean(axis=0)

print("Annual total rainfall per county:")
for county, total in zip(counties, annual_totals):
    print(f"{county}: {total} mm")

print("\nAverage rainfall per month across all counties:")
for month, average in zip(months, monthly_averages):
    print(f"{month}: {average:.1f} mm")

Annual total rainfall per county:
Nairobi: 1045 mm
Nakuru: 916 mm
Kisumu: 1371 mm
Mombasa: 1075 mm
Machakos: 811 mm

Average rainfall per month across all counties:
January: 47.0 mm
February: 47.6 mm
March: 91.0 mm
April: 164.4 mm
May: 171.2 mm
June: 67.6 mm
July: 56.6 mm
August: 58.2 mm
September: 64.2 mm
October: 67.8 mm
November: 127.4 mm
December: 80.6 mm


In [6]:
wettest_month_indexes = np.argmax(R, axis=1)
wettest_months = [months[index] for index in wettest_month_indexes]

print("Wettest month per county:")
for county, wettest_month in zip(counties, wettest_months):
    print(f"{county}: {wettest_month}")

Wettest month per county:
Nairobi: April
Nakuru: April
Kisumu: April
Mombasa: May
Machakos: November


In [7]:
dry_mask = R < 50
dry_month_counts = dry_mask.sum(axis=1)

print("Number of dry months per county:")
for county, dry_count in zip(counties, dry_month_counts):
    print(f"{county}: {dry_count}")

Number of dry months per county:
Nairobi: 4
Nakuru: 3
Kisumu: 0
Mombasa: 2
Machakos: 6


In [8]:
consecutive_rainfall = R[:, :-1] + R[:, 1:]

planting_start_indexes = np.argmax(consecutive_rainfall, axis=1)

month_array = np.array(months)

planting_windows = np.char.add(
    np.char.add(month_array[planting_start_indexes], "-"),
    month_array[planting_start_indexes + 1]
)

print("Recommended planting window per county:")
for county, window in zip(counties, planting_windows):
    print(f"{county}: {window}")

Recommended planting window per county:
Nairobi: April-May
Nakuru: April-May
Kisumu: April-May
Mombasa: April-May
Machakos: November-December


In [9]:
fieldnames = [
    "county",
    "annual_total_mm",
    "wettest_month",
    "dry_months",
    "planting_window"
]

with open("rainfall_summary.csv", "w", newline="", encoding="utf-8") as file:
    writer = csv.DictWriter(file, fieldnames=fieldnames)
    writer.writeheader()

    for county, total, wettest, dry_count, window in zip(
        counties,
        annual_totals,
        wettest_months,
        dry_month_counts,
        planting_windows
    ):
        writer.writerow({
            "county": county,
            "annual_total_mm": int(total),
            "wettest_month": str(wettest),
            "dry_months": int(dry_count),
            "planting_window": str(window)
        })

print("rainfall_summary.csv created successfully.")


rainfall_summary.csv created successfully.


## Planting Advisory

Rainfall analysis indicates that April-May is the strongest two-month planting window for Nairobi, Nakuru, Kisumu, and Mombasa, while November-December is the strongest window for Machakos. Kisumu records the highest annual rainfall total at 1,371 mm and has no months below 50 mm, indicating the most consistent rainfall among the five counties. Machakos records six dry months, the highest number among the counties, so planting should be carefully timed around its recommended window. Nairobi, Nakuru, and Mombasa also experience dry months, making the identified high-rainfall windows important for planting decisions. The April and September rainfall values for Machakos were imputed using the rounded average of its valid readings, so the Machakos recommendation should be interpreted with this data-quality limitation in mind.

## Data Quality Notes

The Machakos rainfall data contained two invalid readings: `N/A` for April and a blank value for September. Both invalid readings were replaced with 68 mm, which is the rounded average of the ten valid Machakos rainfall values. This imputation allowed the dataset to maintain a complete 12-month record for every county. However, the imputed values may not represent the actual rainfall recorded in April and September. The analysis is based only on the supplied monthly rainfall observations and does not account for longer-term seasonal variability.